# 08 · Layer diagnostics — mỗi block của RAD-DINO học được gì?

Chọn block 10–11 cho adapter ở `07` dựa trên **prior chung về ViT**, không phải đo đạc trên
RAD-DINO. Notebook này thay suy đoán đó bằng số liệu.

## Bốn phép đo, bốn câu hỏi khác nhau

| # | Phép đo | Trả lời |
|---|---|---|
| **C3** | **Linear probe từng block** | **Thông tin phục vụ 13 nhãn nằm ở block nào? Bão hòa ở đâu?** ⭐ |
| C4 | Attention distance | Chuyển cục bộ → toàn cục xảy ra ở block nào? |
| C5 | Độ đổi representation giữa 2 block liền kề | Block nào biến đổi mạnh, block nào chỉ tinh chỉnh? |
| C6–C7 | PCA-RGB + attention map (37×37) | Nhìn tận mắt: block sớm bám texture, block muộn bám giải phẫu? |

**C3 là công cụ quyết định.** Đường cong mAP theo độ sâu nói thẳng:

| dạng đường cong | hàm ý cho vị trí adapter |
|---|---|
| tăng đều tới block 12 | thông tin còn đang hình thành ở cuối → block 10–11 **hợp lý** |
| bão hòa ở block 8 | block 9–12 không thêm gì → 10–11 **quá muộn**, nên xuống 7–8 |
| đạt đỉnh rồi giảm | block cuối đang **vứt bỏ** thông tin → càng nên can thiệp sớm |

## Vì sao ViT vẽ được như CNN

1369 patch token chính là lưới **37×37** (518 ÷ 14). Reshape lại là ra ảnh.

Khác CNN ở chỗ: CNN co dần độ phân giải (224→112→…→7) nên xem trực tiếp từng kênh được.
ViT giữ **nguyên `[B, 1370, 768]` ở mọi block** — không có "cô đọng" về hình học. Cái thay
đổi là **mức trừu tượng ngữ nghĩa**, nên ta phải chiếu 768 chiều xuống thứ nhìn được
(PCA→RGB) hoặc xem model **nhìn đâu** (attention).

## Subsample có phân tầng

Không chạy full dataset. Nhưng lấy ngẫu nhiên thuần sẽ cho quá ít ca dương ở nhãn hiếm
(`Pleural_Other` 1.19% ⇒ 8k study chỉ có ~95 ca). Nên C1 **đảm bảo tối thiểu `MIN_POS` ca
dương cho MỖI nhãn**, cộng một phần nền ngẫu nhiên.

> ⚠️ Vì oversample nhãn hiếm, **prevalence bị lệch khỏi tự nhiên**, nên giá trị mAP tuyệt
> đối ở đây **không so được** với 0.4336 của H5. Chỉ **hình dạng đường cong** (block nào cao
> hơn block nào) là có ý nghĩa — và đó đúng là thứ ta cần.


## C0 · Config

In [ ]:
import os, json, math, pickle, random, shutil, tarfile
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
LOCAL_IMG = '/content/cxr_local'   # o tam cua VM, XOA moi lan restart -> C0-bis re-stage
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')
DG_DIR    = os.path.join(DATA_DIR, 'layer_diag'); os.makedirs(DG_DIR, exist_ok=True)

MIN_POS      = 400     # ca dương tối thiểu MỖI nhãn, trong mỗi tập
N_BACKGROUND = 4000    # thêm nền ngẫu nhiên để có âm tính thực tế
N_ATTN_IMG   = 16      # ảnh dùng cho thống kê attention (nặng bộ nhớ)
W_F, W_L     = 0.7, 0.3
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '|', torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU')

# --- palette: validated (6/6 checks pass, light surface) ---
PAL = {'macro':'#0072B2', 'head':'#D55E00', 'medium':'#009E73', 'tail':'#7B3294'}
# C_GRID = mau luoi. KHONG dat ten GRID: C1 dung GRID cho kich thuoc luoi patch (37),
# va trung ten se lam C4/C6 chet voi 'arange() not supported for StrDType' neu C0 chay lai.
INK, MUTED, C_GRID = '#1a1a1a', '#666666', '#e6e6e6'
plt.rcParams.update({'figure.dpi':120, 'font.size':10, 'axes.edgecolor':MUTED,
                     'axes.labelcolor':INK, 'text.color':INK, 'xtick.color':MUTED,
                     'ytick.color':MUTED, 'axes.spines.top':False, 'axes.spines.right':False,
                     'grid.color':C_GRID, 'grid.linewidth':0.8, 'figure.facecolor':'white'})

## C0-bis - Stage anh ve o local

Anh nam tren Drive dang **tar shard** (`cxr_560_shards`); `/content` bi xoa moi lan VM
restart. Cell nay phai chay **trong phien nay**, ke ca khi 04b/05/06b/07 da stage o phien
truoc - runtime moi luon bat dau trong.

**CPU/network, khong can GPU** - ~8 phut cho bo day du, tuc thi neu o con am.

In [ ]:
os.makedirs(LOCAL_IMG, exist_ok=True)
tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar'))
assert tars, f'no shards in {SHARD_DIR} -> run 04a first'
todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
print(f'shards: {len(tars)} on Drive | {len(todo)} to stage')
for f in tqdm(todo, desc='staging shards', unit='shard'):
    tmp = f'/content/{f}'
    shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
    with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
    os.remove(tmp)
    open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()   # marker AFTER extract
N_LOCAL = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
print(f'local images: {N_LOCAL:,}')
assert N_LOCAL > 0, 'staging produced no images'

## C1 · Cohort + subsample phân tầng + backbone

Với **mỗi** nhãn, lấy tối đa `MIN_POS` study dương; hợp lại; thêm nền ngẫu nhiên. Làm riêng
cho `train` (fit probe) và `val` (đánh giá). Bảng in ra cho thấy mỗi nhãn có bao nhiêu ca
dương thực tế — nếu nhãn nào vẫn thiếu thì đó là do cohort không đủ, không phải do sampling.

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR,'cohort_meta.json')))
LABELS = meta['labels']; assert len(LABELS)==13
ehr = pd.read_parquet(os.path.join(OUT_DIR,'cohort_study_level.parquet')).sort_values('study_id').reset_index(drop=True)
cf = pickle.load(open(os.path.join(OUT_DIR,'class_freq.pkl'),'rb'))
TRAIN_PREV = np.asarray(cf['class_freq'])/cf['num_train']
TIER = np.where(TRAIN_PREV<0.02,'tail', np.where(TRAIN_PREV<0.10,'medium','head'))

def stratified(split, min_pos=MIN_POS, n_bg=N_BACKGROUND, seed=SEED):
    """Union of up-to-min_pos positives per label + random background."""
    sub = ehr[ehr.split==split]; rng = np.random.default_rng(seed); keep = set()
    for l in LABELS:
        pos = sub.index[sub[l]==1].to_numpy()
        keep |= set(rng.choice(pos, min(min_pos, len(pos)), replace=False).tolist())
    rest = np.setdiff1d(sub.index.to_numpy(), np.fromiter(keep, int))
    keep |= set(rng.choice(rest, min(n_bg, len(rest)), replace=False).tolist())
    return np.sort(np.fromiter(keep, int))

ROWS = {s: stratified(s) for s in ('train','val')}
rep = pd.DataFrame({'label':LABELS,'tier':TIER,'prev_train':TRAIN_PREV.round(4),
                    'pos_fit':[int(ehr.loc[ROWS['train'],l].sum()) for l in LABELS],
                    'pos_eval':[int(ehr.loc[ROWS['val'],l].sum()) for l in LABELS]}).sort_values('prev_train')
print(f"studies: fit {len(ROWS['train']):,} | eval {len(ROWS['val']):,}")
print(rep.to_string(index=False))
thin = rep[rep.pos_eval < 100]
if len(thin): print(f'\n!! nhãn có <100 ca dương trong eval: {thin.label.tolist()} -> AP của chúng nhiễu')

# ---- image manifest, hạn chế vào các study đã chọn ----
vm = pd.read_parquet(os.path.join(OUT_DIR,'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal','lateral'])]
_dup = [c for c in ('subject_id','split','setting') if c in vm.columns]
man = vm.drop(columns=_dup).merge(ehr[['study_id','split']], on='study_id', how='inner')
_before = len(man)
man = man[man.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))].reset_index(drop=True)
# KHONG loc im lang: anh chua stage -> man rong -> feature rong -> loi no o sklearn cach day 3 cell
assert len(man), (f'ALL {_before:,} images missing from {LOCAL_IMG} -> chay C0-bis truoc '
                  '(100% missing = VM moi, /content da bi xoa)')
assert (_before-len(man))/_before < 0.01, f'{_before-len(man):,}/{_before:,} images missing -> chay lai C0-bis'
SID = {s: set(ehr.loc[ROWS[s],'study_id']) for s in ROWS}
IDX = {s: np.where(man.study_id.isin(SID[s]).to_numpy() & (man.split.to_numpy()==s))[0] for s in ROWS}
for _s, _i in IDX.items():
    assert len(_i) > 1, f'split {_s} chi co {len(_i)} anh -> probe khong chay duoc'
print('\nimages:', {s: len(IDX[s]) for s in IDX})

# ---- backbone ----
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T
from PIL import Image
backbone = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
_sd = torch.load(BB_CKPT, map_location=dev, weights_only=False)['model']
backbone.load_state_dict({k[9:]: v for k,v in _sd.items() if k.startswith('backbone.')}, strict=True)
for p in backbone.parameters(): p.requires_grad = False
LAYERS = backbone.encoder.layer; NB = len(LAYERS)
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
SZ = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
eval_tf = T.Compose([T.CenterCrop(SZ), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
GRID = SZ // backbone.config.patch_size
print(f'\nblocks {NB} | patch grid {GRID}x{GRID} = {GRID*GRID} | hidden {backbone.config.hidden_size}')

class DS(Dataset):
    def __init__(s, idx): s.idx = np.asarray(idx)
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        r = man.iloc[s.idx[k]]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        return eval_tf(Image.merge('RGB',(im,im,im))), s.idx[k]

## C2 · Trích feature ở **cả 12 block** trong một lượt forward

Mỗi block cho `[B, 1370, 768]`. Giữ hết là 12 × 1370 × 768 mỗi ảnh — quá nặng. Nên rút gọn
**ngay tại chỗ** thành `[CLS ‖ mean(patches)]` = 1536 chiều/block. Hai đại lượng này bổ sung
nhau: CLS là tổng hợp toàn cục model tự học, mean-patch là trung bình không thiên vị.

In [ ]:
@torch.no_grad()
def extract(idx, bs=64):
    dl = DataLoader(DS(idx), batch_size=bs, num_workers=8, pin_memory=(dev=='cuda'))
    out = np.zeros((len(idx), NB, 1536), 'float16'); rows = np.zeros(len(idx), 'int64'); p = 0
    for x, r in tqdm(dl, desc='extract', leave=False):
        x = x.to(dev, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=(dev=='cuda')):
            h = backbone.embeddings(x)
            for i in range(NB):
                o = LAYERS[i](h); h = o[0] if isinstance(o,(tuple,list)) else o
                out[p:p+x.shape[0], i] = torch.cat([h[:,0], h[:,1:].mean(1)], -1).float().cpu().numpy()
        rows[p:p+x.shape[0]] = r.numpy(); p += x.shape[0]
    return out, rows

FE = {}
for s in ('train','val'):
    f = f'{DG_DIR}/feat_{s}.npz'
    if os.path.exists(f):
        z = np.load(f); FE[s] = (z['x'], z['rows'])
        if len(z['x']) != len(IDX[s]):     # cache cu/rong -> vut, dung tin
            print(f'{s}: cache co {len(z["x"])} hang nhung IDX co {len(IDX[s])} -> xoa, trich lai')
            os.remove(f); FE[s] = extract(IDX[s]); np.savez(f, x=FE[s][0], rows=FE[s][1])
        else: print(f'{s}: cache')
    else:
        FE[s] = extract(IDX[s]); np.savez(f, x=FE[s][0], rows=FE[s][1])
    print(f'  {s}: {FE[s][0].shape}  ({FE[s][0].nbytes/1e9:.2f} GB fp16)')

## C3 · ⭐ Linear probe từng block — công cụ quyết định

Mỗi block: một `Linear(1536 → 13)` train trên tập fit, chấm trên tập eval **ở mức study**
(pool 7:3, giống pipeline thật). Cùng số bước, cùng LR, cùng chuẩn hóa cho mọi block — nên
so sánh giữa các block là công bằng.

Đọc **hình dạng**, không đọc giá trị tuyệt đối (prevalence đã bị lệch do oversample).

In [ ]:
from sklearn.metrics import average_precision_score
Y_ST = ehr.set_index('study_id')

def pool_eval(p_img, idx):
    sub = man.iloc[idx]
    d = pd.DataFrame(p_img, columns=LABELS)
    d['study_id']=sub.study_id.to_numpy(); d['vg']=sub.view_group.to_numpy()
    sid = np.sort(d.study_id.unique())
    fr = d[d.vg=='frontal'].groupby('study_id')[LABELS].mean().reindex(sid)
    la = d[d.vg=='lateral'].groupby('study_id')[LABELS].mean().reindex(sid)
    hf = fr.notna().all(1).to_numpy('float32'); hl = la.notna().all(1).to_numpy('float32')
    den = (W_F*hf + W_L*hl)[:,None]
    PS = (W_F*np.nan_to_num(fr.to_numpy('float32'))*hf[:,None]
          + W_L*np.nan_to_num(la.to_numpy('float32'))*hl[:,None]) / den
    Yt = Y_ST.loc[sid, LABELS].to_numpy('float32')
    return np.array([average_precision_score(Yt[:,j], PS[:,j]) for j in range(len(LABELS))])

def y_img(idx):
    return ehr.set_index('study_id').loc[man.iloc[idx].study_id, LABELS].to_numpy('float32')

Ytr_i, Yva_i = y_img(IDX['train']), y_img(IDX['val'])
Ytr_t = torch.tensor(Ytr_i, device=dev)

def probe_block(l, steps=600, lr=3e-3, wd=1e-4):
    Xtr = torch.tensor(FE['train'][0][:, l].astype('float32'), device=dev)
    Xva = torch.tensor(FE['val'][0][:, l].astype('float32'), device=dev)
    mu, sd = Xtr.mean(0, keepdim=True), Xtr.std(0, keepdim=True)+1e-6   # fit-set stats only
    Xtr, Xva = (Xtr-mu)/sd, (Xva-mu)/sd
    lin = nn.Linear(Xtr.shape[1], len(LABELS)).to(dev)
    opt = torch.optim.AdamW(lin.parameters(), lr=lr, weight_decay=wd)
    pw = ((len(Ytr_t)-Ytr_t.sum(0))/Ytr_t.sum(0).clamp(min=1)).clamp(max=50)
    for _ in range(steps):
        opt.zero_grad()
        F.binary_cross_entropy_with_logits(lin(Xtr), Ytr_t, pos_weight=pw).backward()
        opt.step()
    with torch.no_grad(): P = torch.sigmoid(lin(Xva)).cpu().numpy()
    del Xtr, Xva; torch.cuda.empty_cache() if dev=='cuda' else None
    return pool_eval(P, IDX['val'])

AP = np.stack([probe_block(l) for l in tqdm(range(NB), desc='probing blocks')])   # [NB, 13]
np.save(f'{DG_DIR}/probe_ap.npy', AP)

curve = pd.DataFrame({'block': np.arange(1, NB+1), 'macro': AP.mean(1).round(4),
                      **{t: AP[:, TIER==t].mean(1).round(4) for t in ('head','medium','tail')}})
print(curve.to_string(index=False))
best = int(AP.mean(1).argmax())
print(f'\nblock tốt nhất (macro): {best+1}  |  mAP {AP.mean(1)[best]:.4f}')
print(f'block 11 (nơi 07 đặt adapter): {AP.mean(1)[10]:.4f}  '
      f'({AP.mean(1)[10]-AP.mean(1)[best]:+.4f} so với đỉnh)')

### Biểu đồ — mAP theo độ sâu

Một trục y duy nhất, 4 đường cùng thang. Nhãn đặt trực tiếp ở cuối đường (không chỉ dựa vào
màu để phân biệt), lưới lùi về sau, đánh dấu vị trí adapter của `07`.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
x = np.arange(1, NB+1)
series = [('macro', AP.mean(1)), ('head', AP[:, TIER=='head'].mean(1)),
          ('medium', AP[:, TIER=='medium'].mean(1)), ('tail', AP[:, TIER=='tail'].mean(1))]
ax.grid(axis='y', zorder=0)
for name, v in series:
    lw = 2.4 if name=='macro' else 1.8
    ax.plot(x, v, color=PAL[name], lw=lw, marker='o', ms=5, zorder=3,
            markeredgecolor='white', markeredgewidth=1.2)
    ax.annotate(name, (x[-1], v[-1]), xytext=(6, 0), textcoords='offset points',
                color=PAL[name], fontweight='600', va='center', fontsize=9)
for b in (11, 12):                                  # adapter blocks của 07 (1-indexed)
    ax.axvline(b, color=MUTED, lw=1, ls=':', zorder=1)
ax.annotate('adapter của 07', (11.5, ax.get_ylim()[0]), xytext=(0, 6),
            textcoords='offset points', ha='center', color=MUTED, fontsize=8)
ax.set_xlabel('block của RAD-DINO'); ax.set_ylabel('study-level AP (subsample phân tầng)')
ax.set_title('Thông tin phục vụ 13 nhãn hình thành ở block nào', pad=12, fontweight='600')
ax.set_xticks(x); ax.set_xlim(0.5, NB+1.4)
plt.tight_layout(); plt.savefig(f'{DG_DIR}/probe_curve.png', dpi=150, bbox_inches='tight'); plt.show()

print('cách đọc:')
print('  tăng đều tới block 12      -> 10-11 hợp lý')
print('  bão hòa ở block ~8         -> 10-11 QUÁ MUỘN, nên xuống 7-8')
print('  đạt đỉnh rồi giảm          -> block cuối vứt bỏ thông tin, can thiệp sớm hơn')

## C4 · Attention distance — chuyển cục bộ → toàn cục

Với mỗi block, khoảng cách không gian trung bình giữa các patch chú ý lẫn nhau (đơn vị:
patch, ×14 = pixel):

$$d_l=\sum_{ij}A^{(l)}_{ij}\,\lVert p_i-p_j\rVert$$

Block sớm ⇒ `d` nhỏ (nhìn quanh mình). Block muộn ⇒ `d` lớn (tích hợp toàn ảnh). **Chỗ `d`
bật lên chính là nơi ngữ nghĩa toàn cục bắt đầu hình thành** — và can thiệp bằng EHR chỉ có
lý sau điểm đó.

In [ ]:
gy, gx = np.meshgrid(np.arange(GRID), np.arange(GRID), indexing='ij')
pos = np.stack([gy.ravel(), gx.ravel()], 1).astype('float32')
DM = torch.tensor(np.linalg.norm(pos[:,None,:]-pos[None,:,:], axis=-1), device=dev)   # [1369,1369]

# SDPA khong vat chat hoa ma tran attention (do la muc dich cua no) -> o.attentions ve
# tuple RONG: vong lap khong chay lan nao, acc giu nguyen 0, KHONG bao loi.
# Nap rieng mot backbone 'eager' chi de xem attention; C1/C2 giu sdpa cho nhanh va nhe.
if 'bb_attn' not in globals():
    bb_attn = AutoModel.from_pretrained('microsoft/rad-dino',
                                        attn_implementation='eager').to(dev).eval()
    _sda = torch.load(BB_CKPT, map_location=dev, weights_only=False)['model']
    bb_attn.load_state_dict({k[9:]: v for k, v in _sda.items() if k.startswith('backbone.')},
                            strict=True)
    for _p in bb_attn.parameters(): _p.requires_grad = False
    del _sda
    print('eager backbone:', bb_attn.config._attn_implementation)

@torch.no_grad()
def _attn_of(x):
    """Attention cua moi block, co chan: tuple rong -> bao loi thay vi tra ve 0."""
    A = bb_attn(pixel_values=x, output_attentions=True).attentions
    assert len(A) == NB, (f'attentions tra ve {len(A)} lop, can {NB} -> '
                          'backbone khong chay eager, con so se toan 0')
    return A

@torch.no_grad()
def attn_distance(n_img=N_ATTN_IMG, bs=2):
    idx = IDX['val'][np.random.default_rng(0).choice(len(IDX['val']), n_img, replace=False)]
    dl = DataLoader(DS(idx), batch_size=bs, num_workers=2)
    acc = np.zeros((NB, backbone.config.num_attention_heads)); n = 0
    for x, _ in tqdm(dl, desc='attention', leave=False):
        att = _attn_of(x.to(dev))
        for l, A in enumerate(att):                 # [B, heads, 1370, 1370]
            a = A[:, :, 1:, 1:].float()                      # patch<->patch only
            a = a / a.sum(-1, keepdim=True).clamp(min=1e-9)
            acc[l] += (a * DM).sum(-1).mean(-1).sum(0).cpu().numpy()
        n += x.shape[0]; del att
        torch.cuda.empty_cache() if dev=='cuda' else None
    return acc / n                                           # [NB, heads]

AD = attn_distance(); np.save(f'{DG_DIR}/attn_distance.npy', AD)
print(pd.DataFrame({'block':np.arange(1,NB+1),'mean_dist_patch':AD.mean(1).round(2),
                    'mean_dist_px':(AD.mean(1)*backbone.config.patch_size).round(1),
                    'head_min':AD.min(1).round(2),'head_max':AD.max(1).round(2)}).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.grid(axis='y', zorder=0)
for h in range(AD.shape[1]):                                  # từng head, mờ
    ax.plot(np.arange(1,NB+1), AD[:,h], color=PAL['medium'], lw=1, alpha=0.25, zorder=2)
ax.plot(np.arange(1,NB+1), AD.mean(1), color=PAL['macro'], lw=2.4, marker='o', ms=5,
        zorder=3, markeredgecolor='white', markeredgewidth=1.2)
ax.annotate('trung bình', (NB, AD.mean(1)[-1]), xytext=(6,0), textcoords='offset points',
            color=PAL['macro'], fontweight='600', va='center', fontsize=9)
ax.annotate('từng head', (NB, AD[-1].min()), xytext=(6,-4), textcoords='offset points',
            color=PAL['medium'], va='center', fontsize=8)
for b in (11,12): ax.axvline(b, color=MUTED, lw=1, ls=':', zorder=1)
ax.set_xlabel('block'); ax.set_ylabel('khoảng cách attention (patch)')
ax.set_title('Attention lan rộng ra toàn ảnh ở block nào', pad=12, fontweight='600')
ax.set_xticks(np.arange(1,NB+1)); ax.set_xlim(0.5, NB+2)
plt.tight_layout(); plt.savefig(f'{DG_DIR}/attn_distance.png', dpi=150, bbox_inches='tight'); plt.show()

## C5 · Representation thay đổi bao nhiêu giữa hai block liền kề

$$1-\cos\big(z^{(l)}_i,\;z^{(l+1)}_i\big)\quad\text{trung bình trên token}$$

Block nào **biến đổi mạnh** (giá trị cao) là nơi đang thực sự tính toán; block nào gần 0 chỉ
đang tinh chỉnh. Chèn adapter vào một block chỉ-tinh-chỉnh thì khó có tác dụng.

In [ ]:
@torch.no_grad()
def block_delta(n_img=64, bs=16):
    idx = IDX['val'][np.random.default_rng(1).choice(len(IDX['val']), n_img, replace=False)]
    dl = DataLoader(DS(idx), batch_size=bs, num_workers=4)
    acc = np.zeros(NB-1); n = 0
    for x, _ in tqdm(dl, desc='delta', leave=False):
        h = backbone.embeddings(x.to(dev)); prev = None
        for i in range(NB):
            o = LAYERS[i](h); h = o[0] if isinstance(o,(tuple,list)) else o
            if prev is not None:
                acc[i-1] += (1 - F.cosine_similarity(prev, h, dim=-1)).mean(-1).sum().item()
            prev = h
        n += x.shape[0]
    return acc / n

BD = block_delta(); np.save(f'{DG_DIR}/block_delta.npy', BD)
fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.grid(axis='y', zorder=0)
ax.bar(np.arange(2, NB+1), BD, color=PAL['macro'], zorder=3, width=0.62)
for b in (11,12): ax.axvline(b, color=MUTED, lw=1, ls=':', zorder=1)
ax.set_xlabel('block (so với block trước)'); ax.set_ylabel('1 − cosine')
ax.set_title('Block nào biến đổi representation mạnh nhất', pad=12, fontweight='600')
ax.set_xticks(np.arange(2, NB+1))
plt.tight_layout(); plt.savefig(f'{DG_DIR}/block_delta.png', dpi=150, bbox_inches='tight'); plt.show()
print(pd.DataFrame({'block':np.arange(2,NB+1),'delta':BD.round(4)}).to_string(index=False))

## C6 · PCA-RGB 37×37 — nhìn tận mắt mỗi block nhóm vùng theo tiêu chí gì

Với mỗi block: lấy 1369 patch token `[1369, 768]`, PCA xuống 3 thành phần, chuẩn hóa thành
RGB, reshape `37×37`. Đây là cách trực quan chuẩn của DINOv2.

**Màu giống nhau = model coi hai vùng đó tương tự nhau.** Block sớm sẽ nhóm theo độ sáng /
texture; block muộn — nếu prior đúng — nhóm theo **cấu trúc giải phẫu** (phổi vs trung thất
vs thành ngực). Nếu block muộn **không** tách theo giải phẫu thì prior của tôi sai.

In [ ]:
@torch.no_grad()
def pca_rgb_grid(dicom_ids, blocks=None):
    blocks = blocks or list(range(NB))
    rows = []
    for did in dicom_ids:
        im = Image.open(f'{LOCAL_IMG}/{did}.jpg').convert('L')
        x = eval_tf(Image.merge('RGB',(im,im,im)))[None].to(dev)
        h = backbone.embeddings(x); maps = []
        for i in range(NB):
            o = LAYERS[i](h); h = o[0] if isinstance(o,(tuple,list)) else o
            if i in blocks:
                P = h[0,1:].float()
                P = (P - P.mean(0)) / (P.std(0)+1e-6)
                V = torch.pca_lowrank(P, q=3)[2][:, :3]
                r = P @ V
                r = (r - r.min(0).values) / (r.max(0).values - r.min(0).values + 1e-9)
                maps.append(r.reshape(GRID, GRID, 3).cpu().numpy())
        rows.append((did, np.array(x[0,0].cpu()), maps))
    return rows

# chọn ca mẫu: 1 ảnh frontal cho vài nhãn dễ nhìn về mặt không gian
SHOW = ['Pneumothorax','Cardiomegaly','Pleural_Effusion','No_Finding']
ex = []
for l in SHOW:
    sid = ehr.loc[ROWS['val']].query(f'`{l}` == 1').study_id.head(1)
    if len(sid)==0: continue
    d = man[(man.study_id==int(sid.iloc[0])) & (man.view_group=='frontal')].dicom_id
    if len(d): ex.append((l, d.iloc[0]))
print('ca mẫu:', [e[0] for e in ex])

SHOW_BLOCKS = [0, 2, 4, 6, 8, 9, 10, 11]
rows = pca_rgb_grid([d for _, d in ex], SHOW_BLOCKS)
fig, axes = plt.subplots(len(rows), len(SHOW_BLOCKS)+1,
                         figsize=(1.55*(len(SHOW_BLOCKS)+1), 1.75*len(rows)))
axes = np.atleast_2d(axes)
for r,(did, img, maps) in enumerate(rows):
    axes[r,0].imshow(img, cmap='gray'); axes[r,0].set_ylabel(ex[r][0], fontsize=8, color=INK)
    if r==0: axes[r,0].set_title('ảnh', fontsize=9, color=MUTED)
    for c,(b,m) in enumerate(zip(SHOW_BLOCKS, maps), start=1):
        axes[r,c].imshow(m)
        if r==0: axes[r,c].set_title(f'blk {b+1}', fontsize=9,
                                     color=PAL['macro'] if b>=10 else MUTED,
                                     fontweight='600' if b>=10 else 'normal')
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
fig.suptitle('PCA-RGB của patch token — màu giống nhau = model coi là cùng loại vùng',
             y=1.005, fontweight='600')
plt.tight_layout(); plt.savefig(f'{DG_DIR}/pca_rgb.png', dpi=150, bbox_inches='tight'); plt.show()

## C7 · Attention map CLS → patch

Model **nhìn đâu** ở mỗi độ sâu. Thang màu `magma` (perceptually uniform, an toàn với mù
màu). Nếu block muộn tập trung vào vùng bệnh lý thì đó là dấu hiệu ngữ nghĩa đã hình thành —
và là chỗ điều kiện hóa bằng EHR có thể tác động.

In [ ]:
@torch.no_grad()
def cls_attention(did):
    im = Image.open(f'{LOCAL_IMG}/{did}.jpg').convert('L')
    x = eval_tf(Image.merge('RGB',(im,im,im)))[None].to(dev)
    att = _attn_of(x)
    out = [A[0,:,0,1:].float().mean(0).reshape(GRID,GRID).cpu().numpy() for A in att]
    del att; torch.cuda.empty_cache() if dev=='cuda' else None
    return np.array(x[0,0].cpu()), out

fig, axes = plt.subplots(len(ex), len(SHOW_BLOCKS)+1,
                         figsize=(1.55*(len(SHOW_BLOCKS)+1), 1.75*len(ex)))
axes = np.atleast_2d(axes)
for r,(lab, did) in enumerate(ex):
    img, att = cls_attention(did)
    axes[r,0].imshow(img, cmap='gray'); axes[r,0].set_ylabel(lab, fontsize=8, color=INK)
    if r==0: axes[r,0].set_title('ảnh', fontsize=9, color=MUTED)
    for c,b in enumerate(SHOW_BLOCKS, start=1):
        a = att[b]; axes[r,c].imshow(a, cmap='magma')
        if r==0: axes[r,c].set_title(f'blk {b+1}', fontsize=9,
                                     color=PAL['macro'] if b>=10 else MUTED,
                                     fontweight='600' if b>=10 else 'normal')
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
fig.suptitle('Attention CLS → patch (sáng = nhìn nhiều)', y=1.005, fontweight='600')
plt.tight_layout(); plt.savefig(f'{DG_DIR}/cls_attention.png', dpi=150, bbox_inches='tight'); plt.show()

## C8 · Kết luận — block nào nên can thiệp

Tổng hợp bốn phép đo thành một khuyến nghị. Đây là **cơ sở đo được** thay cho prior mà `07`
đã dùng.

In [ ]:
macro = AP.mean(1); best = int(macro.argmax())
THR  = 0.005     # nguong 'khong phat bieu duoc' - dung nhat quan voi 06/06b/07
sat  = next((l for l in range(NB) if macro[l] >= macro[best] - THR), best)  # bao hoa
UNIF = float(DM.mean())              # attention DEU tren luoi -> moc so sanh (~19.3 patch)
sel  = AD.mean(1) / UNIF             # <1 = chon loc khong gian ; ->1 = tron deu
loc  = (AD < 0.5*UNIF).sum(1)        # so head con cuc bo o moi block
foc  = int(sel.argmin())             # block chon loc khong gian nhat
early = (macro[min(4, NB-1)] - macro[0]) / (macro[best] - macro[0])
A0, A1 = 10, 11                      # adapter cua 07, 0-indexed

print('===== TONG HOP - ba phep do, ba cau hoi khac nhau =====')
print(f'  [C3] dinh block {best+1} ({macro[best]:.4f}); bao hoa (trong {THR} cua dinh) tu block {sat+1}')
print(f'       {early:.0%} tong muc tang hinh thanh trong 5 block dau')
print(f'       block 10->11->12: {macro[9]:.4f} -> {macro[10]:.4f} -> {macro[11]:.4f}')
print(f'  [C4] moc attention DEU = {UNIF:.1f} patch')
print(f'       chon loc nhat: block {foc+1} ({AD.mean(1)[foc]:.1f} = {sel[foc]:.0%} muc deu)')
print(f'       block {NB}: {AD.mean(1)[-1]:.1f} = {sel[-1]:.0%} muc deu')
print(f'       head con cuc bo (<50% muc deu): b1-6 {loc[:6].tolist()} | b7-12 {loc[6:].tolist()}')
print(f'  [C5] bien doi manh nhat block {int(BD.argmax())+2} ({BD.max():.4f}), '
      f'yeu nhat block {int(BD.argmin())+2} ({BD.min():.4f}) - bien do hep, tin hieu yeu')

print()
print('===== VI TRI ADAPTER - doc theo CA BA, khong chi dinh probe =====')
crit = [
  ('thong tin',      macro[A0] >= 0.995*macro[best],
   f'block {A0+1} o {macro[A0]/macro[best]:.1%} dinh'),
  ('con kha bien',   sat < A0 and macro[A0] - macro[sat] > THR,
   f'block {sat+1}->{A0+1} chi them {macro[A0]-macro[sat]:+.4f} (nguong {THR})'),
  ('chon loc k.gian', sel[A0] < 0.70,
   f'attention o {sel[A0]:.0%} muc deu, {loc[A0]}/{AD.shape[1]} head con cuc bo'),
  ('do sau con lai', (NB-1-A1) >= 2,
   f'sau adapter cuoi con {NB-1-A1} block self-attention'),
]
for nm, ok, why in crit:
    print(f'  [{"DAT  " if ok else "KHONG"}] {nm:16s} {why}')
n_ok = int(sum(bool(ok) for _, ok, _ in crit))   # numpy.bool_ -> int, json khong nuot numpy

print()
if n_ok >= 3:
    print(f'  -> {n_ok}/4 tieu chi dat: block {A0+1}-{A1+1} la vi tri hop ly.')
else:
    print(f'  -> chi {n_ok}/4 tieu chi dat. Block {A0+1}-{A1+1} la noi bieu dien DAY DU nhat')
    print(f'     nhung IT KHA BIEN nhat, attention gan nhu deu, va gan het do sau phia sau.')
    print(f'     Neu kiem lai deep conditioning, vi tri DO DUOC la block '
          f'{min(foc+1, sat)}-{max(foc+1, sat)} (chon loc k.gian nhat -> truoc bao hoa).')

print()
print('  GIOI HAN: dieu nay KHONG lat null cua screen 07. Phep stacking o 06b khong dung')
print('  adapter, khong dung do sau nao, va van null. Phat bieu dung la: screen da kiem')
print('  gia thuyet o cau hinh BAT LOI NHAT cua no - KHONG phai "null la do dat sai cho".')
print()
print('  (subsample phan tang -> prevalence lech khoi tu nhien; chi HINH DANG duong cong')
print('   co nghia, khong so gia tri tuyet doi voi 0.4336 cua H5)')

json.dump(dict(macro=macro.tolist(), best_block=best+1, saturate_block=sat+1,
               block11=float(macro[10]), attn_dist=AD.mean(1).tolist(),
               attn_uniform=UNIF, attn_sel=sel.tolist(), n_local_heads=loc.tolist(),
               focus_block=foc+1, block_delta=BD.tolist(), criteria_passed=n_ok),
          open(f'{DG_DIR}/summary.json','w'), indent=1,
          default=lambda z: z.item() if hasattr(z, 'item') else str(z))   # luoi an toan
print(f'saved -> {DG_DIR}/')